In [1]:
pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
from datetime import date, datetime
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from IPython.display import display

from dotenv import load_dotenv

In [2]:

# Stage 1 – FOUNDATION

load_dotenv()

API_KEY = os.getenv("CH_API_KEY")
if not API_KEY:
    raise RuntimeError("CH_API_KEY not found. Did you create a .env file?")

# ==============================
# 0. CONFIGURATION
# ==============================
BASE_URL = "https://api.company-information.service.gov.uk"

# Example SIC cluster for Stage 1: B2B / consulting / advertising
sic_cluster_b2b_services = [70229, 70221, 73110]


# ==============================
# 1. LOW-LEVEL HTTP HELPER
# ==============================

def make_request(endpoint: str, params: dict | None = None) -> dict:
    """
    Call Companies House API at BASE_URL + endpoint with optional query params.
    Returns the JSON response as a Python dict or raises HTTPError on failure.
    """
    url = f"{BASE_URL}{endpoint}"
    resp = requests.get(
        url,
        auth=(API_KEY, ""),              # API key as username, blank password
        headers={"Accept": "application/json"},
        params=params,
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


# ==============================
# 2. COMPANY-SPECIFIC HELPERS
# ==============================

def fetch_company_profile(company_number: str) -> dict:
    """Get the basic company profile."""
    return make_request(f"/company/{company_number}")


def fetch_filing_history(company_number: str, items_per_page: int = 200) -> dict:
    """Get filing history (list of submitted documents) for a company."""
    params = {"items_per_page": items_per_page, "start_index": 0}
    return make_request(f"/company/{company_number}/filing-history", params=params)


def fetch_charges(company_number: str, items_per_page: int = 200) -> dict:
    """Get list of registered charges (secured lending) for a company."""
    params = {"items_per_page": items_per_page, "start_index": 0}
    return make_request(f"/company/{company_number}/charges", params=params)


# ==============================
# 3. ADVANCED SEARCH (BUILD UNIVERSE)
# ==============================

def advanced_search_companies(
    sic_codes,
    company_status: str = "active",
    size: int = 100,
    start_index: int = 0,
    location: str | None = None,
) -> dict:
    """
    Call /advanced-search/companies filtered by SIC codes (and optionally status, location).
    Returns the JSON response as a Python dict.
    """
    # Convert list of SIC codes -> "70229,70221,73110"
    if isinstance(sic_codes, (list, tuple, set)):
        sic_param = ",".join(str(code) for code in sic_codes)
    else:
        sic_param = str(sic_codes)

    params = {
        "sic_codes": sic_param,
        "company_status": company_status,  # e.g. "active"
        "size": size,                      # how many results to return (max 5000)
        "start_index": start_index,        # paging, start at 0 for now
    }

    if location:
        params["location"] = location  # e.g. "london"

    # Reuse our generic helper
    return make_request("/advanced-search/companies", params=params)


def companies_json_to_df(result_dict: dict) -> pd.DataFrame:
    """
    Convert advanced-search JSON result into a pandas DataFrame
    with the columns we care about.
    Columns: company_number, name, sic_codes, registered_office_address.
    """
    items = result_dict.get("items", [])
    rows = []

    for item in items:
        roa = item.get("registered_office_address") or {}

        # Build a single address string from the parts
        address_parts = [
            roa.get("address_line_1"),
            roa.get("address_line_2"),
            roa.get("locality"),
            roa.get("region"),
            roa.get("postal_code"),
            roa.get("country"),
        ]
        # Keep only non-empty parts and join with ", "
        address = ", ".join(part for part in address_parts if part)

        rows.append(
            {
                "company_number": item.get("company_number"),
                "name": item.get("company_name"),
                "sic_codes": ";".join(item.get("sic_codes") or []),
                "registered_office_address": address,
            }
        )

    return pd.DataFrame(rows)


def build_seed_universe_if_needed(
    sic_codes,
    company_status: str = "active",
    size: int = 100,
    csv_path: Path = Path("data/companies_seed.csv"),
) -> pd.DataFrame:
    """
    If data/companies_seed.csv exists, load and return it.
    Otherwise, call advanced_search_companies to build a seed universe,
    save it to CSV, and return the DataFrame.
    """
    if csv_path.exists():
        print(f"Found existing seed file at {csv_path}, loading it...")
        return pd.read_csv(csv_path)

    print("No seed file found. Calling Companies House API to build one...")

    # 1) Call advanced search
    result = advanced_search_companies(
        sic_codes=sic_codes,
        company_status=company_status,
        size=size,
        start_index=0,
        location=None,  # optionally restrict location here
    )

    print("Total matches reported by API (hits):", result.get("hits"))
    print("Items in this page:", len(result.get("items", [])))

    # 2) Convert to DataFrame
    seed_df = companies_json_to_df(result)

    # 3) Save to CSV
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    seed_df.to_csv(csv_path, index=False)
    print(f"Saved {len(seed_df)} companies to {csv_path}")

    return seed_df


# ==============================
# 4. VERY SIMPLE EXPLORATION
# ==============================

def explore_sample(seed_df: pd.DataFrame, n_sample: int = 10) -> None:
    """
    For n_sample random companies from seed_df:
    - print company number and name
    - print status
    - print number of filings and charges
    """
    n = min(n_sample, len(seed_df))
    sample_df = seed_df.sample(n=n, random_state=42)

    for _, row in sample_df.iterrows():
        company_number = str(row["company_number"])
        company_name = row["name"]

        print("=" * 80)
        print(f"{company_number} – {company_name}")

        try:
            # --- Call the three endpoints ---
            profile = fetch_company_profile(company_number)
            filings = fetch_filing_history(company_number)
            charges = fetch_charges(company_number)

            # --- Extract what we care about ---
            status = profile.get("company_status")
            filings_count = filings.get("total_count", len(filings.get("items", [])))
            charges_count = charges.get("total_count", len(charges.get("items", [])))

            print(f"Status        : {status}")
            print(f"# of filings  : {filings_count}")
            print(f"# of charges  : {charges_count}")

        except Exception as e:
            print("Error fetching data:", e)


def quick_universe_summary(seed_df: pd.DataFrame, n_sample: int = 20) -> pd.DataFrame:
    """
    Take a random sample of companies and build a small summary DataFrame with:
    - company_number
    - status
    - filings_count
    - charges_count
    """
    records = []
    n = min(n_sample, len(seed_df))

    for _, row in seed_df.sample(n=n, random_state=0).iterrows():
        cn = str(row["company_number"])

        try:
            profile = fetch_company_profile(cn)
            filings = fetch_filing_history(cn)
            charges = fetch_charges(cn)

            records.append(
                {
                    "company_number": cn,
                    "status": profile.get("company_status"),
                    "filings_count": filings.get(
                        "total_count", len(filings.get("items", []))
                    ),
                    "charges_count": charges.get(
                        "total_count", len(charges.get("items", []))
                    ),
                }
            )
        except Exception as e:
            print(f"Error fetching data for {cn}: {e}")

    return pd.DataFrame(records)


# ==============================
# 5. MAIN EXECUTION (RUN IN NOTEBOOK)
# ==============================

# Build or load the seed universe for your B2B services cluster
seed_df = build_seed_universe_if_needed(
    sic_codes=sic_cluster_b2b_services,
    company_status="active",
    size=100,
)

print("\n=== Simple exploration for 10 random companies ===")
explore_sample(seed_df, n_sample=10)

print("\n=== Quick summary for 20 companies ===")
summary_df = quick_universe_summary(seed_df, n_sample=20)
display(summary_df)

print("\nStatus counts:")
print(summary_df["status"].value_counts())

print("\nCharges count distribution (top):")
print(summary_df["charges_count"].value_counts().head())

print("\nFilings count summary:")
print(summary_df["filings_count"].describe())


Found existing seed file at data\companies_seed.csv, loading it...

=== Simple exploration for 10 random companies ===
12846804 – AIO APP LIMITED
Status        : active
# of filings  : 16
# of charges  : 0
08800582 – RADHA KRISHNA AGS LTD
Status        : active
# of filings  : 37
# of charges  : 0
11985944 – MY TRAINING RESOURCES LTD
Status        : active
# of filings  : 18
# of charges  : 0
11686882 – JIMMY JAMES COMPANIES UK LTD
Status        : active
# of filings  : 25
# of charges  : 0
11337773 – SHAPE YOUR FUTURE ONLINE LTD
Status        : active
# of filings  : 21
# of charges  : 0
11504523 – DOCTORS IN BUSINESS LTD
Status        : active
# of filings  : 18
# of charges  : 0
10266613 – MESOLEAN GROUP LTD.
Status        : active
# of filings  : 20
# of charges  : 0
15054797 – BAHELMI CONSULTANCY LTD
Status        : active
# of filings  : 7
# of charges  : 0
11797482 – CARNE CONSULTING LIMITED
Status        : active
# of filings  : 8
# of charges  : 0
NI050108 – MCMILLEN CONSULTAN

,company_number,status,filings_count,charges_count
0,12423136,active,14,0
1,11816651,active,20,0
2,12549088,active,17,0
3,SC413867,active,32,0
4,12434325,active,18,0
5,11787455,active,22,0
6,10780140,active,15,0
7,11923073,active,24,0
8,14087296,active,7,0
9,03369943,active,74,2



Status counts:
active    20
Name: status, dtype: int64

Charges count distribution (top):
0    17
2     2
1     1
Name: charges_count, dtype: int64

Filings count summary:
count    20.000000
mean     29.900000
std      23.536981
min       7.000000
25%      16.500000
50%      21.500000
75%      33.250000
max      83.000000
Name: filings_count, dtype: float64


In [3]:
# ==============================
# Stage 2 – FULL PIPELINE (NO src/parsers / 01_parse_companies)
# ==============================

# ---------------------------------
# 0. PATHS & SEED UNIVERSE
# ---------------------------------
project_root = os.getcwd()

DATA_DIR = os.path.join(project_root, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROFILES_DIR = os.path.join(RAW_DIR, "profiles")
FILINGS_DIR = os.path.join(RAW_DIR, "filings")
CHARGES_DIR = os.path.join(RAW_DIR, "charges")

os.makedirs(PROFILES_DIR, exist_ok=True)
os.makedirs(FILINGS_DIR, exist_ok=True)
os.makedirs(CHARGES_DIR, exist_ok=True)

seed_path = os.path.join(DATA_DIR, "companies_seed.csv")
seed_df = pd.read_csv(seed_path, dtype={"company_number": str})

print("Loaded seed universe with", len(seed_df), "companies")
display(seed_df.head())

# ---------------------------------
# 1. CACHE RAW JSON (OPTIONAL BUT NICE)
# ---------------------------------
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

records_done = 0

for _, row in seed_df.iterrows():
    cn = str(row["company_number"])

    profile_path = os.path.join(PROFILES_DIR, f"{cn}.json")
    filings_path = os.path.join(FILINGS_DIR, f"{cn}.json")
    charges_path = os.path.join(CHARGES_DIR, f"{cn}.json")

    # Skip if all three already exist (safe to re-run)
    if (
        os.path.exists(profile_path)
        and os.path.exists(filings_path)
        and os.path.exists(charges_path)
    ):
        print(f"[SKIP] {cn} (already cached)")
        continue

    try:
        profile_json = fetch_company_profile(cn)
        filing_json = fetch_filing_history(cn)
        charges_json = fetch_charges(cn)

        save_json(profile_json, profile_path)
        save_json(filing_json, filings_path)
        save_json(charges_json, charges_path)

        records_done += 1
        print(f"[OK] {cn}")
    except Exception as e:
        print(f"[ERROR] {cn}: {e}")

print(f"Finished. Newly saved JSON for {records_done} companies.")

# ---------------------------------
# 2. FEATURE SCHEMA & EXTRACTORS (Stage 2)
# ---------------------------------
TODAY = date.today()

FEATURE_COLUMNS = [
    # Basic
    "company_number",
    "name",
    "incorporation_date",
    "status",
    "sic_codes",
    "age_years",
    # Filing behaviour
    "num_accounts_last_10y",
    "num_late_accounts",
    "last_accounts_date",
    "overdue_accounts_flag",
    # Charges
    "num_charges_total",
    "num_charges_outstanding",
    "num_charges_satisfied",
    "has_floating_charge_flag",
    "last_charge_date",
]

def parse_date(date_str: str | None):
    """Safe ISO date parser → datetime.date or None."""
    if not date_str:
        return None
    try:
        return datetime.fromisoformat(date_str).date()
    except ValueError:
        return None

def extract_basic_features(profile: dict) -> dict:
    """
    Basic fields:
    - company_number, name, incorporation_date, status, sic_codes, age_years
    """
    incorporation_date = parse_date(profile.get("date_of_creation"))

    sic_codes = profile.get("sic_codes") or []
    if isinstance(sic_codes, (list, tuple, set)):
        sic_str = ";".join(str(code) for code in sic_codes)
    else:
        sic_str = str(sic_codes) if sic_codes else ""

    age_years = None
    if incorporation_date:
        age_years = round((TODAY - incorporation_date).days / 365.25, 2)

    return {
        "company_number": profile.get("company_number"),
        "name": profile.get("company_name"),
        "incorporation_date": incorporation_date,
        "status": profile.get("company_status"),
        "sic_codes": sic_str,
        "age_years": age_years,
    }

def extract_filing_features(profile: dict, filing_history: dict) -> dict:
    """
    Filing behaviour:
    - num_accounts_last_10y
    - num_late_accounts
    - last_accounts_date
    - overdue_accounts_flag (based on next_due < today)
    """
    items = filing_history.get("items", []) or []
    ten_years_ago = date(TODAY.year - 10, TODAY.month, TODAY.day)

    account_dates = []
    num_accounts_last_10y = 0
    num_late_accounts = 0

    for item in items:
        # crude filter for accounts filings
        if item.get("category") != "accounts":
            continue

        d = parse_date(item.get("date"))
        if not d:
            continue

        if d >= ten_years_ago:
            num_accounts_last_10y += 1
            account_dates.append(d)

            desc = (item.get("description") or "").lower()
            if "late" in desc or item.get("is_late"):
                num_late_accounts += 1

    last_accounts_date = max(account_dates) if account_dates else None

    # crude overdue flag from profile.accounts.next_due
    accounts_info = profile.get("accounts") or {}
    next_due = parse_date(accounts_info.get("next_due"))
    overdue_accounts_flag = bool(next_due and next_due < TODAY)

    return {
        "num_accounts_last_10y": num_accounts_last_10y,
        "num_late_accounts": num_late_accounts,
        "last_accounts_date": last_accounts_date,
        "overdue_accounts_flag": overdue_accounts_flag,
    }

def extract_charge_features(charges: dict) -> dict:
    """
    Charges:
    - num_charges_total / outstanding / satisfied
    - has_floating_charge_flag
    - last_charge_date
    """
    items = charges.get("items", []) or []

    num_charges_total = charges.get("total_count", len(items))
    num_charges_satisfied = 0
    num_charges_outstanding = 0
    has_floating_charge_flag = False
    dates = []

    for item in items:
        satisfied_on = parse_date(item.get("satisfied_on"))
        created_on = parse_date(
            item.get("created_on") or item.get("delivered_on")
        )

        if satisfied_on:
            num_charges_satisfied += 1
            dates.append(satisfied_on)
        else:
            # treat missing satisfied_on as still outstanding
            num_charges_outstanding += 1

        if created_on:
            dates.append(created_on)

        classification = item.get("classification") or {}
        if "floating" in (classification.get("type") or "").lower():
            has_floating_charge_flag = True
        if item.get("floating_charge"):
            has_floating_charge_flag = True

    last_charge_date = max(dates) if dates else None

    return {
        "num_charges_total": num_charges_total,
        "num_charges_outstanding": num_charges_outstanding,
        "num_charges_satisfied": num_charges_satisfied,
        "has_floating_charge_flag": has_floating_charge_flag,
        "last_charge_date": last_charge_date,
    }

def build_company_feature_row(company_number: str) -> dict:
    """Call the 3 endpoints and merge all features into a single dict."""
    profile = fetch_company_profile(company_number)
    filings = fetch_filing_history(company_number)
    charges = fetch_charges(company_number)

    features = {}
    features.update(extract_basic_features(profile))
    features.update(extract_filing_features(profile, filings))
    features.update(extract_charge_features(charges))

    return features

# ---------------------------------
# 3. BUILD FEATURE TABLE FOR SEED UNIVERSE
# ---------------------------------
def build_features_for_seed(seed_df: pd.DataFrame,
                            max_companies: int = 200) -> pd.DataFrame:
    records = []

    for _, row in seed_df.head(max_companies).iterrows():
        cn = str(row["company_number"])
        try:
            rec = build_company_feature_row(cn)
            records.append(rec)
        except Exception as e:
            print(f"Error fetching features for {cn}: {e}")

    features_df = pd.DataFrame(records, columns=FEATURE_COLUMNS)
    out_path = os.path.join(DATA_DIR, "company_features_stage2.csv")
    features_df.to_csv(out_path, index=False)
    print("Saved Stage 2 feature table to:", out_path)
    display(features_df.head(100))
    return features_df

# Either reuse an existing stage 2 features CSV, or build it
features_csv_path = os.path.join(DATA_DIR, "company_features_stage2.csv")
if os.path.exists(features_csv_path):
    print("Reusing existing company_features_stage2.csv")
    features_df = pd.read_csv(features_csv_path)
else:
    print("No company_features_stage2.csv found, building from API...")
    features_df = build_features_for_seed(seed_df, max_companies=100)

# Ensure date columns are proper datetimes (not strings)
date_cols = ["incorporation_date", "last_accounts_date", "last_charge_date"]
for col in date_cols:
    if col in features_df.columns:
        features_df[col] = pd.to_datetime(features_df[col], errors="coerce")

# Ensure booleans are booleans
bool_cols = ["overdue_accounts_flag", "has_floating_charge_flag"]
for col in bool_cols:
    if col in features_df.columns:
        features_df[col] = features_df[col].fillna(False).astype(bool)

print("Dtypes after cleaning:")
print(features_df.dtypes)

# ---------------------------------
# 4. ADD SIGNALS (DISTRESS / BORING-PROFITABLE) + SCORES
# ---------------------------------
def add_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Basic safety cleaning ---
    for col in ["num_accounts_last_10y", "num_late_accounts",
                "num_charges_total", "num_charges_outstanding",
                "num_charges_satisfied"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    if "age_years" in df.columns:
        df["age_years"] = df["age_years"].fillna(0)

    if "overdue_accounts_flag" in df.columns:
        df["overdue_accounts_flag"] = df["overdue_accounts_flag"].astype(bool)

    if "has_floating_charge_flag" in df.columns:
        df["has_floating_charge_flag"] = df["has_floating_charge_flag"].astype(bool)

    # --- Distress-like signal (v1 heuristic) ---
    df["signal_potential_distress"] = (
        (df.get("overdue_accounts_flag", False) == True)
        | (df.get("num_late_accounts", 0) >= 1)
        | (df.get("num_charges_outstanding", 0) >= 5)
        | ((df.get("age_years", 0) < 3) & (df.get("num_charges_total", 0) >= 1))
    )

    # --- Boring-but-profitable-like signal (v1 heuristic) ---
    df["signal_boring_profitable_like"] = (
        (df.get("age_years", 0) >= 8)
        & (df.get("num_accounts_last_10y", 0) >= 5)
        & (df.get("num_late_accounts", 0) == 0)
        & (df.get("overdue_accounts_flag", False) == False)
        & (df.get("num_charges_outstanding", 0) <= 1)
    )

    # Optional: very simple numeric scores (0–3)
    df["distress_score_v1"] = (
        df["overdue_accounts_flag"].astype(int)
        + (df["num_late_accounts"] >= 1).astype(int)
        + (df["num_charges_outstanding"] >= 3).astype(int)
    )

    df["boring_profitable_score_v1"] = (
        (df["age_years"] >= 10).astype(int)
        + (df["num_accounts_last_10y"] >= 7).astype(int)
        + (df["num_late_accounts"] == 0).astype(int)
    )

    return df

features_with_signals = add_signals(features_df)

display(
    features_with_signals[[
        "company_number", "name",
        "age_years",
        "num_accounts_last_10y", "num_late_accounts", "overdue_accounts_flag",
        "num_charges_total", "num_charges_outstanding",
        "signal_potential_distress", "distress_score_v1",
        "signal_boring_profitable_like", "boring_profitable_score_v1"
    ]].head(100)
)

# ---------------------------------
# 5. SAVE ENRICHED FEATURES (WITH SIGNALS)
# ---------------------------------
processed_dir = os.path.join(DATA_DIR, "processed")
os.makedirs(processed_dir, exist_ok=True)

features_out_path = os.path.join(
    processed_dir, "company_features_with_signals_v1.csv"
)
features_with_signals.to_csv(features_out_path, index=False)
print("Saved with signals to:", features_out_path)

# ---------------------------------
# 6. SUMMARY STATS ON SIGNALS
# ---------------------------------
print("Shape:", features_with_signals.shape)
print("\nSignal value counts:")
print("Potential distress:")
print(features_with_signals["signal_potential_distress"].value_counts(dropna=False))
print("\nBoring-profitable-like:")
print(features_with_signals["signal_boring_profitable_like"].value_counts(dropna=False))

# ---------------------------------
# 7. BUILD & SAVE WATCHLISTS + SUMMARY COMPANIES
# ---------------------------------
# Top potential distress: sort by distress_score_v1 descending, then youngest first
distress_watchlist = (
    features_with_signals[features_with_signals["signal_potential_distress"] == True]
    .sort_values(
        ["distress_score_v1", "age_years"],
        ascending=[False, True]
    )
)

# Top boring-profitable-like: sort by boring_profitable_score_v1 descending, then oldest first
boring_watchlist = (
    features_with_signals[features_with_signals["signal_boring_profitable_like"] == True]
    .sort_values(
        ["boring_profitable_score_v1", "age_years"],
        ascending=[False, False]
    )
)

# Preview top 10 distress companies
print("\nTop 10 potential distress companies:")
display(
    distress_watchlist[
        ["company_number", "name", "age_years", "distress_score_v1"]
    ].head(10)
)

# Preview top 10 boring-but-profitable companies
print("\nTop 10 boring-but-profitable-like companies:")
display(
    boring_watchlist[
        ["company_number", "name", "age_years", "boring_profitable_score_v1"]
    ].head(10)
)

# Save watchlists
distress_path = os.path.join(processed_dir, "distress_watchlist_v1.csv")
boring_path = os.path.join(processed_dir, "boring_profitable_watchlist_v1.csv")

distress_watchlist.to_csv(distress_path, index=False)
boring_watchlist.to_csv(boring_path, index=False)

print("\nSaved watchlists:")
print("  ", distress_path)
print("  ", boring_path)


Loaded seed universe with 100 companies


,company_number,name,sic_codes,registered_office_address
0,NI050108,MCMILLEN CONSULTANTS LIMITED,70229,"At The Offices Of Falconer Stewart, 248-266 Up..."
1,NI628743,CR FINANCIAL PLANNING LIMITED,70221,"4 Rhanbuoy Gardens, Seahill, Holywood, County ..."
2,12549088,HH SPORTS CONSULTANCY LIMITED,70229,"Milburn House Hexham Business Park, Burn Lane,..."
3,SC119502,MARKET SIGNALS LIMITED,73110,"7 Chapel Road, Strathaven, Lanarkshire, ML10 6NA"
4,12967045,ODIN FINANCE LIMITED,70229,"29 Barnard Road, London, SW11 1QT, England"


[OK] NI050108
[OK] NI628743
[OK] 12549088
[OK] SC119502
[OK] 12967045
[OK] 12973660
[OK] 07795943
[OK] 06708383
[OK] 11768206
[OK] 11772504
[OK] 11797482
[OK] 11626340
[OK] 11924380
[OK] 10373307
[OK] 10207222
[OK] 10080791
[OK] 10780140
[OK] 10795651
[OK] SC629666
[OK] 12578630
[OK] 11430485
[OK] 09999784
[OK] 10266613
[OK] 07183006
[OK] 08140876
[OK] 10845563
[OK] 12423136
[OK] 06267967
[OK] 10728903
[OK] 13995668
[OK] 09423868
[OK] 07740842
[OK] 04448756
[OK] 02773512
[OK] 12023080
[OK] 09220681
[OK] 12199092
[OK] 12251860
[OK] 07707662
[OK] 11504523
[OK] 05180629
[OK] 12563916
[OK] 11579423
[OK] 11582000
[OK] 11337773
[OK] 11686882
[OK] 10180235
[OK] 09750693
[OK] 15664493
[OK] 10937728
[OK] 04425004
[OK] 09762552
[OK] 06590794
[OK] 08800582
[OK] 14087296
[OK] SC413867
[OK] 09418604
[OK] SC587963
[OK] SC585860
[OK] SC262456
[OK] 13836714
[OK] 02954712
[OK] 06670576
[OK] 12401270
[OK] NI645731
[OK] 12490286
[OK] 07725608
[OK] 11826595
[OK] 14100097
[OK] 11917623
[OK] 11985944
[OK] 1

,company_number,name,incorporation_date,status,sic_codes,age_years,num_accounts_last_10y,num_late_accounts,last_accounts_date,overdue_accounts_flag,num_charges_total,num_charges_outstanding,num_charges_satisfied,has_floating_charge_flag,last_charge_date
0,NI050108,MCMILLEN CONSULTANTS LIMITED,2004-03-30,active,70229,21.66,1,0,2016-02-18,True,0,0,0,False,None
1,NI628743,CR FINANCIAL PLANNING LIMITED,2015-01-16,active,70221,10.86,3,0,2018-11-21,True,0,0,0,False,None
2,12549088,HH SPORTS CONSULTANCY LIMITED,2020-04-06,active,70229,5.64,3,0,2023-12-21,True,0,0,0,False,None
3,SC119502,MARKET SIGNALS LIMITED,1989-08-15,active,73110,36.28,9,0,2024-03-22,True,0,0,0,False,None
4,12967045,ODIN FINANCE LIMITED,2020-10-22,active,70229,5.10,0,0,None,True,0,0,0,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,12490286,A J DUNLOP CONSULTING LIMITED,2020-02-28,active,70229,5.74,5,0,2025-10-24,False,0,0,0,False,None
66,07725608,LANDWISE RURAL RESOURCE MANAGEMENT LIMITED,2011-08-02,active,70229,14.32,10,0,2025-04-03,False,0,0,0,False,None
67,11826595,ESSEX ADVERTISING SERVICES LTD,2019-02-14,active,73110,6.78,1,0,2021-02-13,True,0,0,0,False,None
68,14100097,SYSTEMS4FOOD LTD,2022-05-11,active,70229,3.55,3,0,2025-07-14,False,0,0,0,False,None


Dtypes after cleaning:
company_number                      object
name                                object
incorporation_date          datetime64[ns]
status                              object
sic_codes                           object
age_years                          float64
num_accounts_last_10y                int64
num_late_accounts                    int64
last_accounts_date          datetime64[ns]
overdue_accounts_flag                 bool
num_charges_total                    int64
num_charges_outstanding              int64
num_charges_satisfied                int64
has_floating_charge_flag              bool
last_charge_date            datetime64[ns]
dtype: object


...

Saved with signals to: c:\Users\adgg875\OneDrive - City St George's, University of London\Project 101\Week 2\data\processed\company_features_with_signals_v1.csv
Shape: (70, 19)

Signal value counts:
Potential distress:
True     44
False    26
Name: signal_potential_distress, dtype: int64

Boring-profitable-like:
False    54
True     16
Name: signal_boring_profitable_like, dtype: int64

Top 10 potential distress companies:


,company_number,name,age_years,distress_score_v1
60,13836714,HC DEBT MANAGEMENT LTD,3.88,1
5,12973660,BEECHWAY FINANCIAL LTD,5.08,1
4,12967045,ODIN FINANCE LIMITED,5.10,1
19,12578630,QUBO CONSULTANTS LTD,5.58,1
41,12563916,KABOKY LTD,5.60,1
2,12549088,HH SPORTS CONSULTANCY LIMITED,5.64,1
26,12423136,KPMUK LTD,5.84,1
63,12401270,TOP KEK LTD,5.87,1
37,12251860,BOULEVARD MARKETING LTD,6.13,1
36,12199092,AGENT ELITES LTD,6.21,1



Top 10 boring-but-profitable-like companies:


,company_number,name,age_years,boring_profitable_score_v1
33,02773512,RUPERT TAYLOR LIMITED,32.95,3
61,02954712,CRYSTAL DESIGN SERVICES LIMITED,31.32,3
50,04425004,AZTEC CONSULTANTS (GB) LIMITED,23.59,3
40,05180629,NUCLEUS COMPLETE BUSINESS ADVICE LIMITED,21.37,3
52,06590794,ICJR CONSULTANCY LIMITED,17.54,3
38,07707662,THE FULL HOUSE FOR FAMILIES LTD,14.36,3
66,07725608,LANDWISE RURAL RESOURCE MANAGEMENT LIMITED,14.32,3
31,07740842,DECOUPLE LIMITED,14.28,3
24,08140876,THE WE TECHNOLOGY (UK) LTD,13.37,3
53,08800582,RADHA KRISHNA AGS LTD,11.98,3



Saved watchlists:
   c:\Users\adgg875\OneDrive - City St George's, University of London\Project 101\Week 2\data\processed\distress_watchlist_v1.csv
   c:\Users\adgg875\OneDrive - City St George's, University of London\Project 101\Week 2\data\processed\boring_profitable_watchlist_v1.csv
